# LeetCode #197: Rising Temperature

https://leetcode.com/problems/rising-temperature/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Self-Join with DATEDIFF ★ | O(n²) | O(result) |
| Window Function (LAG) | O(n log n) | O(n) |

## Understanding the Methods
### Self-Join with DATEDIFF (Optimal SQL)
Join the Weather table with itself where the date difference is exactly 1 day, and the temperature on the later date is higher. Return the id of the later date.

### Window Function (LAG)
Use LAG to access the previous day's temperature, then filter rows where current temperature exceeds the previous. Requires ordering by date.

## Solutions

### C#

In [ ]:
// SQL Problem — SQL solution:
/*
SELECT w1.id
FROM Weather w1
JOIN Weather w2
ON DATEDIFF(w1.recordDate, w2.recordDate) = 1
WHERE w1.temperature > w2.temperature;
*/

// C# equivalent using LINQ:
using System;
using System.Collections.Generic;
using System.Linq;

var weather = new List<(int id, DateTime date, int temp)> {
    (1, new DateTime(2015, 1, 1), 10),
    (2, new DateTime(2015, 1, 2), 25),
    (3, new DateTime(2015, 1, 3), 20),
    (4, new DateTime(2015, 1, 4), 30)
};

var result = from w1 in weather
             from w2 in weather
             where (w1.date - w2.date).Days == 1 && w1.temp > w2.temp
             select w1.id;

foreach (var id in result)
    Console.WriteLine(id);

### Python

In [ ]:
import pandas as pd

def rising_temperature(weather: pd.DataFrame) -> pd.DataFrame:
    """
    SQL equivalent:
    SELECT w1.id FROM Weather w1
    JOIN Weather w2 ON DATEDIFF(w1.recordDate, w2.recordDate) = 1
    WHERE w1.temperature > w2.temperature;
    """
    weather = weather.sort_values('recordDate')
    weather['prev_temp'] = weather['temperature'].shift(1)
    weather['prev_date'] = weather['recordDate'].shift(1)
    mask = (
        (weather['recordDate'] - weather['prev_date']).dt.days == 1
    ) & (weather['temperature'] > weather['prev_temp'])
    return weather.loc[mask, ['id']]

### Go

In [ ]:
package main

import (
    "fmt"
    "time"
)

type Weather struct {
    ID          int
    RecordDate  time.Time
    Temperature int
}

func risingTemperature(records []Weather) []int {
    dateMap := make(map[string]Weather)
    for _, r := range records {
        dateMap[r.RecordDate.Format("2006-01-02")] = r
    }
    var result []int
    for _, r := range records {
        prev := r.RecordDate.AddDate(0, 0, -1).Format("2006-01-02")
        if p, ok := dateMap[prev]; ok && r.Temperature > p.Temperature {
            result = append(result, r.ID)
        }
    }
    return result
}

func main() {
    records := []Weather{
        {1, time.Date(2015, 1, 1, 0, 0, 0, 0, time.UTC), 10},
        {2, time.Date(2015, 1, 2, 0, 0, 0, 0, time.UTC), 25},
        {3, time.Date(2015, 1, 3, 0, 0, 0, 0, time.UTC), 20},
        {4, time.Date(2015, 1, 4, 0, 0, 0, 0, time.UTC), 30},
    }
    fmt.Println(risingTemperature(records))
}

### Rust

In [ ]:
use std::collections::HashMap;

#[derive(Debug, Clone)]
struct Weather { id: i32, date: String, temperature: i32 }

fn rising_temperature(records: &[Weather]) -> Vec<i32> {
    let date_map: HashMap<&str, &Weather> = records.iter()
        .map(|r| (r.date.as_str(), r)).collect();
    
    records.iter().filter_map(|r| {
        // Simple date arithmetic for YYYY-MM-DD format
        let parts: Vec<i32> = r.date.split('-')
            .map(|s| s.parse().unwrap()).collect();
        let prev = format!("{:04}-{:02}-{:02}", parts[0], parts[1], parts[2] - 1);
        // Note: simplified; real code should handle month/year boundaries
        if let Some(p) = date_map.get(prev.as_str()) {
            if r.temperature > p.temperature {
                return Some(r.id);
            }
        }
        None
    }).collect()
}

fn main() {
    let records = vec![
        Weather { id: 1, date: "2015-01-01".into(), temperature: 10 },
        Weather { id: 2, date: "2015-01-02".into(), temperature: 25 },
        Weather { id: 3, date: "2015-01-03".into(), temperature: 20 },
        Weather { id: 4, date: "2015-01-04".into(), temperature: 30 },
    ];
    println!("{:?}", rising_temperature(&records));
}

## Examples

**Common:** Dates 1-4 with temps [10,25,20,30] → ids [2,4] — day 2 warmer than day 1, day 4 warmer than day 3.

**Slightly Complex:** Consecutive days with temps [30,20,25,35] → ids for days 3 and 4 (25>20, 35>25).

**Edge Case (Always Falling):** Temps [30,20,10] → empty result.

**Edge Case (Non-Consecutive):** Dates Jan 1 and Jan 3 (gap) → Jan 3 not compared to Jan 1.

**Single Row:** Only one weather record → empty result (no previous day to compare).

![Infographic](attachment:image.png)